# Week 1 Lab, Part 1: Python and numpy on a real trajectory

You will load a molecular dynamics trajectory, find where a small molecule (the *ligand*) is in every frame, and plot how far it moves. In Part 2 you will check whether that plot means what it seems to mean.

**How to work**

* Run a cell with **Shift+Enter**. Cells run in order, top to bottom.
* **Predict first** cells: write your guess *before* you run the code. It does not have to be right. Writing down what you expect before you run a result is the habit the course policy asks for, and it is how you notice results that are implausible.
* Cells marked **TODO** need your code. Cells whose first line is `# check` must run without an error before you move on; they are your tests.
* You may use an AI assistant under the course policy. Keep your AI-use log as you go: what you asked, what was wrong, what you changed, and where it helped your understanding or got in the way. Your predictions and your checks are graded on your own reasoning.
* **Units.** `mdtraj` stores coordinates in **nanometers (nm)**. One nanometer is 10 angstroms (Å). A variable name that ends in `_nm` holds nanometers and `_A` holds angstroms.

### Setup: is this the course environment?

Run the next cell. The `python` line should point into the course environment (on ICER that path starts with `/mnt/research/BMB_961_SS27_001/env`). If it points somewhere else, you are using the wrong kernel: ask for help before going on.

In [ ]:
import os, sys, platform
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt
import mdtraj as md

# Folder that holds sEH_nowater.pdb and sEH_unbinding.dcd: the ML4MD_DATA variable if it is set, otherwise this folder.
DATA_DIR = Path(os.environ.get("ML4MD_DATA", "."))
print("python :", platform.python_version(), "from", sys.executable)
print("numpy  :", np.__version__, "| mdtraj:", md.__version__)
print("data   :", "found" if (DATA_DIR / "sEH_unbinding.dcd").exists() and (DATA_DIR / "sEH_nowater.pdb").exists() else "NOT FOUND in " + str(DATA_DIR.resolve()))

## 1. Python: the few things you need

If you already program, skim this section and do the TODOs. If you do not, this is everything you need for today.

### 1.1 Names, numbers and units

A **name** is a label for a value. Python does not remember units, so put the unit in the name.

In [ ]:
x_nm = 0.52            # a distance in nanometers
x_A = x_nm * 10        # 1 nm = 10 angstroms
print(f"{x_nm} nm = {x_A} A")

**TODO.** A carbon-carbon single bond is about 1.5 Å long. Convert it to nanometers and store it in `bond_nm`.

In [ ]:
bond_A = 1.5
bond_nm = None   # TODO: replace None with your conversion

# check
assert abs(bond_nm - 0.15) < 1e-12, "1.5 angstroms should be 0.15 nm"

### 1.2 Lists and loops

A **list** holds values in order. Counting starts at **0**. A slice `a[1:3]` includes position 1 and stops *before* position 3.

**Predict first.** For `x_nm = [0.10, 0.25, 0.50]`, what do `x_nm[0]`, `x_nm[-1]` and `x_nm[1:3]` give?

My prediction: 

In [ ]:
x_nm = [0.10, 0.25, 0.50]
print(x_nm[0], x_nm[-1], x_nm[1:3])

x_A = []                     # a loop repeats an indented block for each item
for x in x_nm:
    x_A.append(x * 10)
print(x_A)

**TODO.** Write a loop that adds up the values in `x_nm` and stores the result in `total_nm`.

In [ ]:
total_nm = 0.0
# TODO: add a for loop here

# check
assert abs(total_nm - 0.85) < 1e-12, "0.10 + 0.25 + 0.50 is 0.85"

### 1.3 Functions

A **function** packages a calculation so you can reuse it. The distance between two points $\mathbf{a} = (a_x, a_y, a_z)$ and $\mathbf{b} = (b_x, b_y, b_z)$ is

$$
d = \sqrt{(b_x - a_x)^2 + (b_y - a_y)^2 + (b_z - a_z)^2}
$$

**In words:** take the difference in each direction, square each, add the three, then take the square root. This is Pythagoras' theorem in three dimensions. The result has the same unit as the coordinates (nm in, nm out).

**Worked example.** For $\mathbf{a} = (0, 0, 0)$ and $\mathbf{b} = (3, 4, 12)$: the differences are 3, 4 and 12; the squares are 9, 16 and 144; their sum is 169; and $\sqrt{169} = 13$.

**Two ways to write it.** The same formula can be coded in two ways, and you will write both:

* with plain Python and `math.sqrt`: square the three differences, add them, and take the square root of the sum;
* with numpy: the difference $\mathbf{b} - \mathbf{a}$ is itself a vector, and `np.linalg.norm` returns the length of a vector, which is exactly the square root above. A tuple cannot be subtracted from a tuple, so turn each point into an array first with `np.asarray`.

`math.sqrt` takes one number at a time. `np.linalg.norm` takes a whole array and, with its `axis` argument, can measure many vectors at once; section 2.6 uses that. The two functions must give the same number, so the last lines of the cell below check that they do.

In [ ]:
import math

def distance_math(a, b):
    """Distance between points a and b, each a sequence (x, y, z), using math.sqrt. Same unit in and out."""
    # TODO: replace the next line with the calculation above, using math.sqrt
    return None

def distance_numpy(a, b):
    """The same distance, using np.linalg.norm on the difference vector b - a."""
    # TODO: replace the next line: convert a and b with np.asarray, subtract, and take the norm
    return None

# check
for f in (distance_math, distance_numpy):
    assert f((0, 0, 0), (3, 4, 12)) == 13.0, f"{f.__name__}: the worked example above should give 13"
    assert abs(f((0, 0, 0), (0.3, 0.4, 1.2)) - 1.3) < 1e-12, f"{f.__name__}: same shape in nm should give 1.3 nm"
p, q = (0.1, -0.2, 0.3), (1.1, 0.9, -0.4)
assert abs(distance_math(p, q) - distance_numpy(p, q)) < 1e-12, "the two ways should agree"

### 1.4 Getting help

You will look things up all semester. In Jupyter, `help(name)` prints the documentation, `name?` shows it in a pane, and **Tab** completes names. The part to read first is the **Parameters** section.

**TODO.** Run `help(np.linalg.norm)` and read what `axis` does. Then answer in your own words in the cell below it: for an array of shape `(N, 3)`, what does `np.linalg.norm(a, axis=1)` return, and what is its shape?

In [ ]:
help(np.linalg.norm)

**My answer:** 

**Log it.** If an assistant helped with anything above, write one line in your AI-use log now.

## 2. numpy on the sEH trajectory

**numpy** gives Python fast arrays and operations on whole arrays at once. The data: *soluble epoxide hydrolase* (sEH) with a small-molecule inhibitor, residue name `2RV`, leaving its binding pocket. A **frame** is one saved snapshot of all atom positions; frames are numbered 0, 1, 2, and so on.

### 2.1 Load the data

The PDB file lists every atom (names, residues, elements). The DCD file holds the coordinates of those atoms in every frame.

In [ ]:
pdb_file = DATA_DIR / "sEH_nowater.pdb"
dcd_file = DATA_DIR / "sEH_unbinding.dcd"
traj = md.load(dcd_file, top=pdb_file)
print(traj)

### 2.2 The coordinate array

`traj.xyz` is a numpy array with one number for every frame, atom and direction (x, y, z), in nm.

**Predict first.** What is the shape of `traj.xyz`? (The cell above printed the number of frames and atoms.)

My prediction: 

In [ ]:
xyz = traj.xyz
print("shape :", xyz.shape, "= (frames, atoms, x/y/z)")
print("dtype :", xyz.dtype, f"-> {xyz.itemsize} bytes per number")
print(f"memory: {xyz.nbytes / 1e6:.1f} MB")

# check: the memory is (number of values) x (bytes per value)
assert xyz.nbytes == traj.n_frames * traj.n_atoms * 3 * xyz.itemsize

### 2.3 Indexing and slicing

Each index picks a position along one axis; a colon `:` keeps the whole axis. First a tiny array, so you can see it.

In [ ]:
a = np.arange(12).reshape(3, 4)     # 3 rows, 4 columns, filled 0..11
print(a)
print("a[1]     :", a[1])           # row 1
print("a[:, 1]  :", a[:, 1])        # column 1, every row
print("a[1, 2]  :", a[1, 2])        # row 1, column 2
print("a[:2, 1:3]:", a[:2, 1:3].tolist())   # rows 0-1, columns 1-2

**Predict first.** For each expression, write the shape you expect in the dictionary below (a tuple like `(3, 4)`), *then* run the cell. A wrong guess is fine: it tells you where your picture of the array differs from numpy's.

In [ ]:
predicted = {
    "xyz[0]":        None,   # TODO: replace each None with your predicted shape
    "xyz[0, 0]":     None,
    "xyz[:5]":       None,
    "xyz[:, 0, :]":  None,
    "xyz[:, :, 2]":  None,
}
actual = {k: eval(k).shape for k in predicted}
for k in predicted:
    print(f"{k:14s} predicted {str(predicted[k]):16s} actual {str(actual[k]):16s}", "same" if predicted[k] == actual[k] else "DIFFERENT")

**In your own words:** what does `xyz[:, 0, :]` contain?

My answer: 

### 2.4 Selecting atoms

The topology knows the names. `traj.topology.select(...)` returns the **integer indices** of the atoms that match a query. The ligand has residue name `2RV`.

**Predict first.** How many atoms does `traj.topology.select("resname 2RV")` return?

My prediction: 

In [ ]:
lig = traj.topology.select("resname 2RV")
print("atoms selected:", len(lig))

Zero atoms, and no error message. A selection that silently matches nothing is a classic way to get a wrong answer that *looks* fine. The query language reads `2RV` as the number 2 followed by letters. **TODO.** Quote the residue name (`"resname '2RV'"`) and select again.

In [ ]:
lig = None   # TODO: select the ligand atoms with a query that works

# check
assert lig is not None and len(lig) > 0, "no atoms selected: is the residue name quoted?"
print("ligand atoms:", len(lig), "| first indices:", lig[:5])

**Indexing with an array.** Putting an array of indices in one slot picks those atoms: `xyz[:, lig, :]` keeps every frame, only the ligand atoms, and all three directions.

**Boolean masks.** A comparison on an array gives an array of `True`/`False`, and a mask can pick elements. Here we find the **heavy atoms** (everything except hydrogen). The symbol `~` means "not".

In [ ]:
lig_xyz = xyz[:, lig, :]
print("lig_xyz shape:", lig_xyz.shape)

is_hydrogen = np.array([atom.element.symbol == "H" for atom in traj.topology.atoms])   # one True/False per atom in the file
lig_heavy = lig[~is_hydrogen[lig]]                                                       # ligand atoms that are not hydrogen
print("ligand atoms:", len(lig), "| heavy atoms:", len(lig_heavy))

**Check against chemistry.** The ligand's formula is C16 H20 F3 N3 O3. **TODO.** From the formula, compute the total number of atoms and the number of non-hydrogen atoms, and check them against the numbers above.

In [ ]:
# formula C16 H20 F3 N3 O3
n_total_formula = None    # TODO: add up all the atoms in the formula
n_heavy_formula = None    # TODO: add up the atoms that are not hydrogen

# check
assert n_total_formula == len(lig), "total atom count does not match the formula"
assert n_heavy_formula == len(lig_heavy), "heavy-atom count does not match the formula"

### 2.5 Reducing along an axis: the ligand's center

The **geometric center** (centroid) of $N$ atoms is the average of their positions:

$$
\mathbf{r}_c = \frac{1}{N} \sum_{i=1}^{N} \mathbf{r}_i
$$

where $\mathbf{r}_i = (x_i, y_i, z_i)$ is the position of atom $i$ (nm) and $\mathbf{r}_c$ is the center (nm). **In words:** average the x values, average the y values, average the z values.

**Worked example.** Three atoms at $(0,0,0)$, $(3,0,0)$ and $(0,3,0)$ have the center $\left(\tfrac{0+3+0}{3}, \tfrac{0+0+3}{3}, 0\right) = (1, 1, 0)$.

**What you will compute.** The position of the ligand's center in every frame: a table with one row per frame (1107 rows) and three columns, x, y and z, in nm.

**What goes in and what comes out.** Two arrays are involved, and it helps to keep their shapes in mind:

| name | what it holds | shape |
|:--|:--|:--|
| `lig_xyz` | the positions of the 45 ligand atoms in every frame (nm) | `(1107, 45, 3)` |
| `centroid_nm` | the ligand center in every frame (nm): **one** (x, y, z) per frame | `(1107, 3)` |

Going from `lig_xyz` to `centroid_nm` replaces the 45 atoms in each frame by their average. The atoms axis (length 45) disappears, and the other two axes stay.

In [ ]:
three = np.array([[0, 0, 0], [3, 0, 0], [0, 3, 0]], dtype=float)
print(three.mean(axis=0))      # axis=0: average down the rows, one result per column

**Which axis disappears?** `mean(axis=k)` averages along axis `k`, and that axis is the one that disappears from the shape. The cell below builds a tiny stand-in for `lig_xyz`: 2 frames, 4 atoms (the corners of a square) and 3 directions. Run it and read the printed shapes to see which axis is which.

In [ ]:
# A tiny stand-in for lig_xyz: 2 frames, 4 atoms (the corners of a square), 3 directions (x, y, z).
tiny = np.array([
    [[0, 0, 0], [2, 0, 0], [0, 2, 0], [2, 2, 0]],      # frame 0: the four corners
    [[1, 1, 1], [3, 1, 1], [1, 3, 1], [3, 3, 1]],      # frame 1: the same square, moved by (1, 1, 1)
], dtype=float)
print("tiny.shape :", tiny.shape, "= (frames, atoms, x/y/z)")
print()
for axis in (0, 1, 2):
    print(f"tiny.mean(axis={axis}).shape = {tiny.mean(axis=axis).shape}")
print()
print("tiny.mean(axis=1) =")
print(tiny.mean(axis=1))
print("one row per frame: the center of the four atoms in that frame")

In [ ]:
# The real data, described the same way.
print("lig_xyz.shape      :", lig_xyz.shape, "= (frames, ligand atoms, x/y/z)")
print("lig_xyz[0].shape   :", lig_xyz[0].shape, "= frame 0: one row per ligand atom")
print("first three ligand atoms in frame 0 (x, y, z in nm):")
print(lig_xyz[0, :3, :])

**Predict first.** You want `centroid_nm` to have the shape `(1107, 3)`. Compare that with `lig_xyz.shape`, which is `(1107, 45, 3)`: which number disappeared, and so which `axis` do you average over?

My prediction: 

In [ ]:
centroid_nm = None   # TODO: the average of lig_xyz over the atoms axis, in nm

# check 1: the shape is (frames, 3)
assert centroid_nm is not None, "replace None above with your calculation"
print("centroid_nm.shape:", centroid_nm.shape, "| wanted:", (traj.n_frames, 3))
assert centroid_nm.shape == (traj.n_frames, 3), "wrong shape: which axis of lig_xyz has to disappear?"
# check 2: frame 0 computed a second way, with a plain Python loop over the ligand atoms
total = np.zeros(3)
for i in lig:
    total += xyz[0, i, :]
assert np.allclose(centroid_nm[0], total / len(lig), atol=1e-6), "frame 0 disagrees with the loop"
print("centroid in frame 0 (nm):", centroid_nm[0])

### 2.6 Broadcasting: how far from where it started?

**What you will compute.** One number for every frame: the straight-line distance between the ligand center in that frame and the ligand center in frame 0. In frame 0 that distance is 0. In later frames it grows as the ligand drifts away from where it started. You will plot these 1107 numbers below.

**Worked example.** Say the center is at (1, 1, 0) nm in frame 0 and at (4, 5, 0) nm in frame 2. The arrow from the first point to the second is (4 - 1, 5 - 1, 0 - 0) = (3, 4, 0) nm. Its length is $\sqrt{3^2 + 4^2 + 0^2} = 5$ nm, which is 50 Å. So the number for frame 2 is 50 (in Å) and the number for frame 0 is 0.

**Which arrays you need.** Only the ligand center, `centroid_nm`, which already has **one point per frame**. You do not need `lig_xyz` here: its 45 atoms per frame have already been averaged into that one point. (If you subtract from `lig_xyz` instead, you get one arrow per *atom* per frame, shape `(1107, 45, 3)`, which is a different quantity.)

| name | what it holds | shape |
|:--|:--|:--|
| `centroid_nm` | the ligand center in every frame (nm) | `(1107, 3)` |
| `centroid_nm[0]` | the ligand center in frame 0 (nm): a single point | `(3,)` |
| `disp_nm` | for every frame, the **arrow** from the frame-0 center to that frame's center (nm) | `(1107, 3)` |
| `d_A` | the **length** of each arrow, in angstroms | `(1107,)` |

Subtracting `centroid_nm[0]` from `centroid_nm` has to subtract the same three numbers from each of the 1107 rows. numpy lines the shapes up **from the right**, stretches the missing axis, and does exactly that. This is **broadcasting**. First a small example.

In [ ]:
demo = np.array([[1., 2., 3.], [4., 5., 6.], [7., 8., 9.]])      # shape (3, 3)
print("demo[0]        :", demo[0], "  shape", demo[0].shape)
print("demo - demo[0] :")
print(demo - demo[0])         # the shape (3,) row is subtracted from each of the three rows

**Step 1: one frame by hand.** Before doing all 1107 frames, do one. The cell below takes the ligand center in frame 0 (`start`) and in frame 5 (`later`), subtracts them to get the arrow, and measures its length. Run it and read each printed line. (`name = expression` stores the result of the expression under that name, and `centroid_nm[5]` is row 5 of the array.)

In [ ]:
start = centroid_nm[0]       # the ligand center in frame 0: three numbers (x, y, z), in nm
later = centroid_nm[5]       # the ligand center in frame 5
arrow = later - start        # the arrow from start to later: three numbers, in nm
print("start (frame 0):", start, "nm")
print("later (frame 5):", later, "nm")
print("arrow           :", arrow, "nm      shape", arrow.shape)
print("length of arrow :", np.linalg.norm(arrow), "nm, which is", 10 * np.linalg.norm(arrow), "A")
print("by the formula of section 1.3 :", np.sqrt(arrow[0]**2 + arrow[1]**2 + arrow[2]**2), "nm")

**Step 2: all frames at once.** Step 1 did frame 5. To do every frame, subtract `start` from the **whole** array `centroid_nm` instead of from `centroid_nm[5]`. The result, `disp_nm`, has one arrow per row, so its shape is `(1107, 3)`, and its row 5 is the `arrow` you just printed.

In [ ]:
disp_nm = None   # TODO: centroid_nm minus the frame-0 center (start), for all frames at once

# check
assert disp_nm is not None, "replace None above with your calculation"
print("disp_nm.shape:", disp_nm.shape, "| wanted:", (traj.n_frames, 3))
assert disp_nm.shape == (traj.n_frames, 3), "wrong shape: disp_nm is built from centroid_nm, which has one row per frame"
assert np.allclose(disp_nm[0], 0), "in frame 0 the arrow must be zero"
assert np.allclose(disp_nm[5], centroid_nm[5] - centroid_nm[0]), "row 5 should equal the arrow from Step 1"
print("row 5 of disp_nm:", disp_nm[5], "(the same as the arrow in Step 1)")

**Step 3: the length of each arrow.** The length of one arrow $\Delta\mathbf{r}_t = \mathbf{r}_c(t) - \mathbf{r}_c(0)$ (nm) is the distance formula from section 1.3:

$$
d_t = \sqrt{\Delta x_t^2 + \Delta y_t^2 + \Delta z_t^2}
$$

In Step 1, `np.linalg.norm(arrow)` on **one** arrow (shape `(3,)`) gave one number. `disp_nm` holds 1107 arrows stacked as rows, shape `(1107, 3)`, and you want 1107 lengths, shape `(1107,)`: the axis of length 3 (x, y, z) has to disappear. Pass that `axis` to `np.linalg.norm`, as in section 1.4. The lengths come out in nm; multiply by 10 to get angstroms (1 nm = 10 Å).

**Predict first.** What must `d_A[0]` be, and why?

My prediction: 

In [ ]:
d_A = None       # TODO: the length of each arrow in disp_nm, in angstroms (the lengths in nm, times 10)

# check
assert d_A is not None, "replace None above with your calculation"
print("d_A.shape:", d_A.shape, "| wanted:", (traj.n_frames,))
assert d_A.shape == (traj.n_frames,), "wrong shape: one length per frame; which axis of disp_nm has to disappear?"
assert d_A[0] == 0.0, "the displacement in frame 0 must be zero"
assert abs(d_A[5] - 10 * np.sqrt(((centroid_nm[5] - centroid_nm[0])**2).sum())) < 1e-4, "frame 5 disagrees with the length of the arrow from Step 1"
print("frame 5:", round(float(d_A[5]), 3), "A (Step 1 printed the same length)")
print("largest displacement:", round(float(d_A.max()), 2), "A")

**Plot it.** A figure is a claim, so label both axes with what they are **and their units**. Course style: **no title on the figure** (say what it shows in the text around it), and axis labels in **sentence case**: the first word capitalized and the rest lower case, like an ordinary sentence. **TODO.** Fill in the two labels. The check requires a capital first letter, a unit in parentheses on the y axis, and no title.

In [ ]:
frames = np.arange(traj.n_frames)
fig, ax = plt.subplots(figsize=(6.5, 3.8))
ax.plot(frames, d_A)
ax.set_xlabel("")   # TODO: what is on the x axis? Sentence case, with units if it has any
ax.set_ylabel("")   # TODO: what is on the y axis? Sentence case, with the unit in parentheses
plt.show()          # no title: the text around a figure says what it shows

# check
def sentence_case(label):
    words = label.split("(")[0].split()          # ignore the unit in parentheses
    if not label[:1].isupper():
        return False
    first, rest = words[0], words[1:]
    first_ok = first.isupper() or first[1:].islower() or len(first) == 1     # "Frame", or an acronym such as "RMSD"
    return first_ok and all(w.islower() or w.isupper() or not w[0].isalpha() for w in rest)
assert sentence_case(ax.get_xlabel()) and sentence_case(ax.get_ylabel()), "sentence case: a capital first letter, then lower-case words (acronyms such as RMSD and units may be capitalized)"
assert "frame" in ax.get_xlabel().lower(), "label the x axis"
assert "(" in ax.get_ylabel() and ")" in ax.get_ylabel(), "put the unit in parentheses in the y label, for example (A)"
assert ax.get_title() == "", "figures in this course have no title"

### 2.7 Boolean masks on results: when does it first move far?

A comparison gives a mask, `np.where(mask)[0]` gives the positions where it is `True`, and `mask.sum()` counts them.

In [ ]:
far = d_A > 10                          # True where the center is more than 10 A from its start
frames_far = np.where(far)[0]
print("frames beyond 10 A:", int(far.sum()), "of", len(far))
print("first such frame  :", int(frames_far[0]))

**TODO.** Find the first frame where the displacement exceeds **5 Å**, and store it as an integer in `first_5A`. Report it as a frame number, with the threshold.

In [ ]:
first_5A = None   # TODO

# check
assert isinstance(first_5A, (int, np.integer))
assert d_A[first_5A] > 5 and (first_5A == 0 or d_A[first_5A - 1] <= 5), "this is not the FIRST frame beyond 5 A"
print(f"the center first moves more than 5 A from frame 0 at frame {first_5A}")

*A note on numbering.* numpy and mdtraj count frames from 0. Some programs (VMD, for one) count from 1, so the same frame has a number one larger there. Always say which convention you used.

### 2.8 Loops versus whole-array operations

**What we are timing.** In section 2.6 you computed, for each of the 1107 frames, how far the ligand's center is from where it was in frame 0. That is **1107 distances**, one per frame, and it is one task. There are two ways to get the same 1107 numbers:

1. **A Python loop.** Go through the frames one at a time. For frame `i`, call a small function, `point_distance()`, on two points: the center in frame 0 and the center in frame `i`. It applies the distance formula of section 1.3 with `math.sqrt` and is given to you in the cell below. That is 1107 function calls, each doing a tiny calculation.
2. **One numpy operation.** Subtract the frame-0 center from the whole `(1107, 3)` array of centers at once, and take the length of every row with `np.linalg.norm(..., axis=1)`. This is the calculation you wrote in section 2.6. numpy does the repeating in compiled code, so no Python loop runs.

Both give the same 1107 numbers (the cell checks that). Only the **time the computer takes** differs. The cell uses a stopwatch function, `best_time`: it runs a function 5 times and reports the **fastest** run, because slower runs include interruptions from other programs on the node. Times are printed in **milliseconds** (1 ms = 0.001 s). The last line is a ratio: the loop time divided by the numpy time, so "50 times as long" means the loop takes 50 times longer than numpy.

**The speed depends on the computer** (ICER nodes differ a lot), so record **your own** numbers. Your neighbor's will differ.

**Predict first.** For 1107 frames, will the numpy version be about 2, 30 or 1000 times faster than the loop?

My prediction: 

In [ ]:
import math, time

def best_time(fn, repeat=5):
    """The stopwatch: run fn() `repeat` times and return the fastest run, in seconds."""
    best = float("inf")
    for _ in range(repeat):
        t0 = time.perf_counter(); fn(); best = min(best, time.perf_counter() - t0)
    return best

def point_distance(a, b):
    """The distance formula of section 1.3 for two points (given, so this cell does not depend on your own functions)."""
    return math.sqrt((b[0] - a[0])**2 + (b[1] - a[1])**2 + (b[2] - a[2])**2)

# Way 1, the loop: one call of point_distance() per frame, 1107 calls in all.
def loop_version():
    return np.array([point_distance(centroid_nm[0], centroid_nm[i]) for i in range(len(centroid_nm))])

# Way 2, one numpy operation on the whole (1107, 3) array of centers.
def numpy_version():
    return np.linalg.norm(centroid_nm - centroid_nm[0], axis=1)

t_loop, t_numpy = best_time(loop_version), best_time(numpy_version)
print(f"loop : {t_loop * 1e3:8.3f} ms   (1107 calls of point_distance())")
print(f"numpy: {t_numpy * 1e3:8.4f} ms   (one operation on the whole array)")
print(f"the loop takes {t_loop / t_numpy:.0f} times as long as numpy on this machine")

# check: both versions give the same distances (in nm)
assert np.allclose(loop_version(), numpy_version(), atol=1e-6)

**Stretch (optional): a harder case.** Now the quantity is different. For every frame, take the **average over all 5104 atoms** of how far each atom has moved from its frame-0 position. That is 1107 frames times 5104 atoms, about 5.65 million distances. Again there are two ways:

1. **One frame at a time:** a loop over the 1107 frames, with numpy doing the 5104 atoms inside each frame.
2. **All at once:** one operation on the whole `(1107, 5104, 3)` array. It first creates a temporary array as large as `xyz` (about 68 MB), which costs time and memory too.

Which is faster on your node? Write one sentence on why you think so.

**Extra question.** In the loop above, `point_distance` uses `math.sqrt`. Before you try it, predict: if the loop called your `distance_numpy` from section 1.3 instead, would it get faster or slower, and why?

In [ ]:
# Way 1: one frame at a time (a loop over 1107 frames; numpy handles the 5104 atoms in each frame).
def mean_shift_by_frame(xyz):
    return np.array([np.linalg.norm(xyz[i] - xyz[0], axis=1).mean() for i in range(xyz.shape[0])])

# Way 2: everything at once on the whole (1107, 5104, 3) array.
def mean_shift_whole_array(xyz):
    return np.linalg.norm(xyz - xyz[0], axis=2).mean(axis=1)

t_a, t_b = best_time(lambda: mean_shift_by_frame(xyz), repeat=3), best_time(lambda: mean_shift_whole_array(xyz), repeat=3)
print(f"one frame at a time: {t_a * 1e3:7.1f} ms | whole array at once: {t_b * 1e3:7.1f} ms")
assert np.allclose(mean_shift_by_frame(xyz), mean_shift_whole_array(xyz), atol=1e-5)

**Your sentence:** 

### 2.9 Broadcasting in two directions: how big is the ligand?

Sanity checks catch unit mistakes. A drug-like molecule of 25 heavy atoms should be about a nanometer or two long. To measure it, compute the distance between **every pair** of ligand atoms in frame 0 and take the largest.

Put the 45 positions in a column `(45, 1, 3)` and in a row `(1, 45, 3)`. Subtracting them broadcasts to `(45, 45, 3)`: every atom minus every other atom. The norm over the last axis gives a `(45, 45)` table of distances.

In [ ]:
pos = xyz[0, lig, :]                      # (45, 3): the ligand in frame 0, in nm
pair_nm = None                            # TODO: the (45, 45) table of pairwise distances, in nm
# hint: pos[:, None, :] has shape (45, 1, 3) and pos[None, :, :] has shape (1, 45, 3)

# check
assert pair_nm.shape == (len(lig), len(lig))
assert np.allclose(np.diag(pair_nm), 0), "an atom is 0 away from itself"
assert np.allclose(pair_nm, pair_nm.T), "the distance from i to j equals the distance from j to i"
assert abs(pair_nm[3, 7] - np.sqrt(((pos[3] - pos[7])**2).sum())) < 1e-5, "disagrees with the distance formula applied to atoms 3 and 7"
print(f"longest atom-to-atom distance in the ligand: {10 * pair_nm.max():.1f} A")

**Is it plausible?** Compare the number above with the 1.5 Å carbon-carbon bond from section 1.1, and say in one sentence what that tells you about the units.

My answer: 

### 2.10 Saving and loading arrays

Save your displacement array so you can use it again, and **check that what you read back is what you wrote**.

In [ ]:
np.save("ligand_displacement_A.npy", d_A)                       # binary, exact
np.savetxt("ligand_displacement_A.csv", np.c_[frames, d_A],      # text, readable anywhere
           delimiter=",", header="frame,displacement_A", fmt=["%d", "%.4f"], comments="")

back_npy = np.load("ligand_displacement_A.npy")
back_csv = np.loadtxt("ligand_displacement_A.csv", delimiter=",", skiprows=1)

# check
assert np.array_equal(back_npy, d_A), "the .npy file should give back exactly what was saved"
assert np.allclose(back_csv[:, 1], d_A, atol=1e-4) and back_csv.shape == (traj.n_frames, 2)
print("saved and reloaded:", back_npy.shape, back_csv.shape)

### What to keep

* This notebook with your **predictions**, your **checks** passing, and the plot from section 2.6.
* The one-line **PROOF** output from your ICER job (see the ICER onboarding page).
* A start on your **AI-use log**: entry #1.

**Part 2 begins with a question about the plot you just made:** the ligand center moved, but moved *relative to what*, and what else in this file is moving at the same time? Do not answer yet.